# ASL Citizen Dataset Audit (Kaggle)

Audits the attached ASL Citizen mirror and produces the versioned report that `docs/DATA_CONTRACT.md` requires before full training.

This runs on Kaggle rather than Colab because the mirror is already attached read-only at `/kaggle/input`, so there is no download step. The audit is read-only and its outputs are small. See D-007.

The audit never modifies the dataset.

**Before running:** attach the ASL Citizen dataset via *Add Data*, and enable internet access in the notebook settings so the repository can be cloned.

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/Adgonzalez2018/ASL-Recognition-Model.git"

!git clone -q $REPO_URL /kaggle/working/asl
PROJECT = "/kaggle/working/asl/ASL_training"

!pip install -q -e $PROJECT --no-deps
!pip install -q av

## 2. Find the attached dataset

Record the exact mirror being audited. A hosted copy must not be assumed identical to the official release just because it shares a name.

In [ ]:
import os

inputs = sorted(os.listdir("/kaggle/input"))
print("Attached datasets:", inputs)

assert inputs, "No dataset attached. Use Add Data in the notebook sidebar."

# Set this explicitly if more than one dataset is attached.
DATASET_DIR = inputs[0]
DATASET_ROOT = f"/kaggle/input/{DATASET_DIR}"
SOURCE_ID = DATASET_DIR  # recorded in the audit report as the mirror identity

print(f"\nAuditing: {DATASET_ROOT}")

## 3. Inspect the layout first

Resolves split files, the video directory, and the annotation columns, then stops. Confirm these look right before running the full audit — a misread column would mean auditing the wrong thing.

In [ ]:
!cd $PROJECT && python scripts/audit_dataset.py \
    --dataset-root $DATASET_ROOT --layout-only

## 4. Quick structural check

Parses annotations, builds the label map, validates split integrity, and probes a sample of videos. Fast, and catches most structural problems.

This is a **partial** audit and is labeled as such in the report. It is not sufficient for a full training run.

In [ ]:
!cd $PROJECT && python scripts/audit_dataset.py \
    --dataset-root $DATASET_ROOT \
    --output-dir /kaggle/working/artifacts \
    --probe-limit 500 \
    --expected-classes 2731 \
    --source-id $SOURCE_ID

## 5. Full audit

Probes every video and writes the manifests and label map. Slow — it opens every file — but a complete audit is what a real baseline requires.

A non-zero exit means integrity errors were found. Those block training.

In [ ]:
!cd $PROJECT && python scripts/audit_dataset.py \
    --dataset-root $DATASET_ROOT \
    --output-dir /kaggle/working/artifacts \
    --write-manifests \
    --expected-classes 2731 \
    --configured-frames 16 \
    --source-id $SOURCE_ID

## 6. Review the findings

The numbers below are what the dataset actually contains. Compare them against the official ASL Citizen publication before treating any run as a valid baseline.

In [ ]:
import json

with open("/kaggle/working/artifacts/audits/asl_citizen_audit.json") as handle:
    report = json.load(handle)

counts = report["counts"]
print(f"records   {counts['manifest_records']}")
print(f"classes   {counts['classes']}")
print(f"signers   {counts['signers']}")
print(f"by split  {counts['by_split']}")
print(f"signers   {counts['signers_by_split']}")

print(f"\nlabel map identity {report['label_map_identity']}")
print(f"manifest identity  {report['manifest_identity']}")

print("\nclass balance:")
print(json.dumps(report["class_balance"], indent=2))

media = {k: v for k, v in report["media"].items() if k != "resolutions"}
print("\nmedia:")
print(json.dumps(media, indent=2))
print("\nresolutions:", report["media"]["resolutions"])

if report["problems"]:
    print(f"\n{len(report['problems'])} PROBLEM(S):")
    for problem in report["problems"]:
        print(f"  - {problem}")
else:
    print("\nNo problems found.")

## 7. Questions the audit must answer

Check these against the report before Phase 2C:

1. Does the class count match the official 2731? If not, why?
2. Are the splits signer-independent? `integrity.signer_independent` must be true.
3. How many videos are missing or unreadable, and are they concentrated in one split or class?
4. How short is the shortest clip? This determines the short-video policy.
5. How imbalanced are the classes? This shapes macro F1 expectations.
6. Do any videos carry rotation metadata? Decoders disagree on applying it.
7. Is there any handedness or mirroring metadata? If not, no flip policy can be justified yet.

Commit the audit report to the repository. It is small, and it is the evidence that a later run was trained on a known dataset.